In [0]:
gold = spark.read.table("workspace.tfl.silver_journeys")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

gold_daily_hires = (
    gold.groupBy(gold["start_timestamp"].cast(DateType()).alias("date"))
    .agg(
        F.count("*").alias("total_hires"),
        (F.avg("total_duration_ms") / 60000).alias("avg_ride_duration_minutes"),
    )
    .orderBy("date")
)

gold_daily_hires.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.tfl.gold_daily_hires")

spark.sql("""
    COMMENT ON TABLE workspace.tfl.gold_daily_hires
    IS 'Daily count of total TfL cycle hires and their average journey time.'
""")

In [0]:
from pyspark.sql import functions as F

starts = (
    gold.groupBy(
        F.monthname("start_timestamp").alias("month"),
        F.col("start_station_name").alias("station_name")
    )
    .agg(F.count("*").alias("total_hires_started"))
)
ends = (
    gold.groupBy(
        F.monthname("end_timestamp").alias("month"),
        F.col("end_station_name").alias("station_name")
    )
    .agg(F.count("*").alias("total_hires_ended"))
)

gold_station_activity = (
    starts.join(ends, ["month", "station_name"], "full_outer")
    .fillna(0, subset=["total_hires_started", "total_hires_ended"])
    .orderBy("station_name")
)        

gold_station_activity.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.tfl.gold_station_activity")

spark.sql("""
    COMMENT ON TABLE workspace.tfl.gold_station_activity
    IS 'Total amount of jounreys beginning and ending at each station.'
""")

In [0]:

from pyspark.sql import functions as F

gold_popular_routes = (
    gold.groupBy("start_station_name", "end_station_name")
    .agg(
        F.count("*").alias("total_journeys"),
        (F.avg("total_duration_ms") / 60000).alias("avg_ride_duration_minutes"),
    )
    .orderBy(F.desc("total_journeys"))
)

gold_popular_routes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.tfl.gold_popular_routes")

spark.sql("""
    COMMENT ON TABLE workspace.tfl.gold_popular_routes
    IS 'Most popular routes between two stations and their average journey time.'
""")